# Per-Channel Optimal Clustering from 20D Embeddings

This notebook reuses the original 20-dimensional video embeddings and searches for the best cluster count per channel (k from **4 to 10**) to maximize the same performance correlation metric used previously (**Adjusted R²** of `log1p(view_count) ~ C(cluster_id)`).

## 1) Setup and inputs

In [1]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import statsmodels.formula.api as smf

from google.colab import drive
drive.mount('/content/drive')

# Input artifact must contain original 20D embeddings for each video (not 2D clustered export).
DATA_PATH = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/business_cluster_video_embeddings_reduced_20d.csv')

# Search space for channel-specific clustering.
MIN_K = 4
MAX_K = 5
MIN_VIDEOS_PER_CHANNEL = 10
RANDOM_STATE = 42
N_INIT = 20

OUTPUT_DIR = Path('/content/drive/MyDrive/Graphiko/exports/video_embeddings_clustered/optimized')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON = OUTPUT_DIR / 'business_cluster_video_embeddings_channel_optimized_reduced.json'
OUTPUT_METRICS_CSV = OUTPUT_DIR / 'channel_optimal_clustering_metrics.csv'


Mounted at /content/drive


## 2) Load data and resolve embedding columns

In [8]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        'Expected 20D embeddings artifact at: '
        f'{DATA_PATH.resolve()}\n' # Fix: Removed newline from inside f-string and added explicit \n
        'Use the reduced 20D export (for example: '
        '/content/drive/MyDrive/Graphiko/exports/video_embeddings_reduced/latest/'
        'business_cluster_video_embeddings_reduced_20d.csv).'
    )

if DATA_PATH.suffix.lower() == '.csv':
    df = pd.read_csv(DATA_PATH)
elif DATA_PATH.suffix.lower() == '.json':
    df = pd.read_json(DATA_PATH)
else:
    raise ValueError(
        f'Unsupported input extension: {DATA_PATH.suffix}. '
        'Expected .csv or .json containing 20D embeddings.'
    )

required_base = {'channel_name', 'video_id', 'video_title', 'view_count'}
missing = required_base - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

# Detect 20D embeddings schema and reject 2D-only artifacts with a specific message.
embedding_col = None
dim_cols = None

if 'embedding_20d' in df.columns:
    embedding_col = 'embedding_20d'
elif 'embedding' in df.columns and not df['embedding'].dropna().empty:
    sample = df['embedding'].dropna().iloc[0]
    if hasattr(sample, '__len__') and len(sample) == 20:
        embedding_col = 'embedding'
    elif hasattr(sample, '__len__') and len(sample) == 2:
        raise ValueError(
            'Detected 2D embeddings in `embedding`. This notebook requires original 20D embeddings. '
            'Do not use `business_cluster_video_embeddings_clustered_2d.json`; '
            'use `business_cluster_video_embeddings_reduced_20d.csv` instead.'
        )

if embedding_col is None:
    # Modified: Recognize 'embedding_reduced_XX' columns as scalar dimensions
    scalar_dim_cols = [c for c in df.columns if c.startswith('embedding_reduced_') and c[len('embedding_reduced_'):].isdigit()]
    if len(scalar_dim_cols) >= 20:
        # Sort based on the numeric suffix to ensure correct order
        dim_cols = sorted(scalar_dim_cols, key=lambda c: int(c[len('embedding_reduced_'):]))[:20]
    elif len(scalar_dim_cols) == 2:
        raise ValueError(
            'Detected only 2 scalar embedding dimensions (`embedding_0`, `embedding_1`). '
            'This input is 2D, but this notebook requires 20D embeddings.'
        )

if embedding_col is None and dim_cols is None:
    available_embedding_cols = [c for c in df.columns if c.startswith('embedding')]
    raise ValueError(
        'Could not find a valid 20D embeddings schema. Expected one of: '
        '`embedding_20d` (length-20 vectors), `embedding` (length-20 vectors), '
        'or 20 scalar columns `embedding_0`..`embedding_19`. '
        f'Found embedding-like columns: {available_embedding_cols[:10]}'
    )

df['view_count'] = pd.to_numeric(df['view_count'], errors='coerce')
df = df.dropna(subset=['channel_name', 'view_count']).copy()
df['log_views'] = np.log1p(df['view_count'])

print(f'Rows loaded: {len(df):,}')
print(f'Channels: {df["channel_name"].nunique():,}')
print(f'Embedding source: {embedding_col if embedding_col else dim_cols[:3] + ["..."]}')

Rows loaded: 1,344
Channels: 27
Embedding source: ['embedding_reduced_01', 'embedding_reduced_02', 'embedding_reduced_03', '...']


## 3) Helpers: embedding extraction + scoring

In [9]:

def extract_matrix(channel_df: pd.DataFrame) -> np.ndarray:
    if embedding_col is not None:
        mat = np.vstack(channel_df[embedding_col].apply(np.array).to_numpy())
    else:
        mat = channel_df[dim_cols].to_numpy(dtype=float)
    if mat.shape[1] != 20:
        raise ValueError(f'Expected 20 embedding dimensions, got {mat.shape[1]}')
    return mat

def score_adj_r2(channel_df: pd.DataFrame, cluster_labels: np.ndarray) -> float:
    tmp = channel_df.copy()
    tmp['cluster_id'] = pd.Series(cluster_labels, index=tmp.index).astype(str)
    model = smf.ols('log_views ~ C(cluster_id)', data=tmp).fit()
    return float(model.rsquared_adj)


## 4) Per-channel optimization (k = 4..10)

In [13]:

optimized_rows = []
channel_metrics = []

for channel_name, g in df.groupby('channel_name', sort=True):
    g = g.copy().reset_index(drop=True)
    n = len(g)

    if n < max(MIN_VIDEOS_PER_CHANNEL, MIN_K):
        channel_metrics.append({
            'channel_name': channel_name,
            'n_videos': n,
            'best_k': np.nan,
            'best_adj_r2': np.nan,
            'eligible': False,
            'note': 'Insufficient videos'
        })
        continue

    X = extract_matrix(g)
    max_k_here = min(MAX_K, n - 1)
    if max_k_here < MIN_K:
        channel_metrics.append({
            'channel_name': channel_name,
            'n_videos': n,
            'best_k': np.nan,
            'best_adj_r2': np.nan,
            'eligible': False,
            'note': 'Insufficient videos for min k'
        })
        continue

    best = {'k': None, 'adj_r2': -np.inf, 'labels': None}

    for k in range(MIN_K, max_k_here + 1):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        labels = km.fit_predict(X)
        adj_r2 = score_adj_r2(g, labels)

        if adj_r2 > best['adj_r2']:
            best = {'k': k, 'adj_r2': adj_r2, 'labels': labels}

    g['cluster_id'] = best['labels'].astype(int)
    g['cluster_name'] = g['cluster_id'].map(lambda cid: f'{channel_name} | Category {cid + 1}')
    g['best_k_for_channel'] = best['k']
    g['best_adj_r2_for_channel'] = best['adj_r2']

    # reduced embedding fields for downstream JSON consumers
    if embedding_col is not None:
        emb = np.vstack(g[embedding_col].apply(np.array).to_numpy())
    else:
        emb = g[dim_cols].to_numpy(dtype=float)
    g['embedding_20d'] = [row.tolist() for row in emb]
    g['x'] = emb[:, 0]
    g['y'] = emb[:, 1]

    optimized_rows.append(g)
    channel_metrics.append({
        'channel_name': channel_name,
        'n_videos': n,
        'best_k': int(best['k']),
        'best_adj_r2': float(best['adj_r2']),
        'eligible': True
    })

optimized_df = pd.concat(optimized_rows, ignore_index=True) if optimized_rows else pd.DataFrame()
metrics_df = pd.DataFrame(channel_metrics).sort_values(['eligible', 'best_adj_r2'], ascending=[False, False])

print(metrics_df)
print(f'Optimized rows: {len(optimized_df):,}')


                       channel_name  n_videos  best_k  best_adj_r2  eligible
18             Real Vision Presents        50       5     0.592269      True
14            Network State Podcast        50       5     0.575118      True
6                           Bg2 Pod        44       5     0.537975      True
2                      Alex Hormozi        50       5     0.331890      True
11                     Joe Lonsdale        50       5     0.329710      True
3                    All-In Podcast        50       5     0.235318      True
23                     Tony Robbins        50       4     0.220490      True
15                    Patrick Boyle        50       4     0.192023      True
16               Peter H. Diamandis        50       4     0.174762      True
24                    Valuetainment        50       4     0.157204      True
5                       Asianometry        50       5     0.150809      True
1                        ARK Invest        50       4     0.126580      True

## 5) Export JSON + metrics

In [12]:

if len(optimized_df) == 0:
    raise RuntimeError('No optimized rows to export.')

export_cols = [
    'channel_name', 'video_id', 'video_title', 'video_url', 'view_count',
    'cluster_id', 'cluster_name', 'best_k_for_channel', 'best_adj_r2_for_channel',
    'embedding_20d', 'x', 'y'
]
existing_export_cols = [c for c in export_cols if c in optimized_df.columns]

optimized_df[existing_export_cols].to_json(OUTPUT_JSON, orient='records', indent=2)
metrics_df.to_csv(OUTPUT_METRICS_CSV, index=False)

print(f'Wrote JSON: {OUTPUT_JSON.resolve()}')
print(f'Wrote metrics CSV: {OUTPUT_METRICS_CSV.resolve()}')


Wrote JSON: /content/drive/MyDrive/Graphiko/exports/video_embeddings_clustered/optimized/business_cluster_video_embeddings_channel_optimized_reduced.json
Wrote metrics CSV: /content/drive/MyDrive/Graphiko/exports/video_embeddings_clustered/optimized/channel_optimal_clustering_metrics.csv
